In [1]:
from pylib.setup import *
setup_notebook()

from pylib.assumptions import demand, smr_tech, SMR_DOWNTIME, solar_tech, wind_tech, battery, GRID_TARIFF_BUY, GRID_TARIFF_SELL, grid_connect_annual
from pylib.model import GridSupply, KKSupply, VESupply, DatacenterModel

## 1. Load inputs
*Three 8760-row txt files from `1_VE.ipynb`. Solar and wind are normalised capacity factors (production / installed capacity). Prices are hourly DK weighted-average spot prices in €/MWh.*

In [2]:
pat = pathlib.Path("variation_patterns")

# Installed capacities from 1_VE.ipynb API output — used to normalise MWh/h → CF [0,1]
_solar_cap_mw = 4_955.5428
_wind_cap_mw  = 4_878.483

solar_cf = np.loadtxt(pat / "PV_VE_2025_2026.txt") / _solar_cap_mw   # MWh/h → CF
wind_cf  = np.loadtxt(pat / "WL_VE_2025_2026.txt") / _wind_cap_mw    # MWh/h → CF
prices   = np.loadtxt(pat / "wp_2025_2026.txt")    / 7.46             # DKK/MWh → EUR/MWh

print(f"solar_cf : {solar_cf.shape}  mean={solar_cf.mean():.3f}  max={solar_cf.max():.3f}")
print(f"wind_cf  : {wind_cf.shape}   mean={wind_cf.mean():.3f}  max={wind_cf.max():.3f}")
print(f"prices   : {prices.shape}    mean={prices.mean():.1f}   min={prices.min():.1f}  max={prices.max():.1f}  €/MWh")

solar_cf : (8760,)  mean=0.103  max=0.750
wind_cf  : (8760,)   mean=0.229  max=0.816
prices   : (8760,)    mean=81.6   min=-30.7  max=583.5  €/MWh


## 2. Demand and grid
*`DatacenterDemand` sets the total load (1 GW flat) and the on-site fraction `x`. `GridSupply` wraps the price series and computes import costs given any on-site dispatch profile.*

In [3]:
grid = GridSupply(prices, demand)

print(f"Total demand   : {demand.demand_mw:,.0f} MW")
print(f"On-site floor  : {demand.floor_mw:,.0f} MW  (x = {demand.x:.0%})")
print(f"Grid cap       : {demand.grid_cap_mw:,.0f} MW")
print(f"Annual demand  : {demand.annual_mwh/1e6:.3f} TWh")

Total demand   : 1,000 MW
On-site floor  : 500 MW  (x = 50%)
Grid cap       : 500 MW
Annual demand  : 8.760 TWh


## 3. KK — nuclear on-site
*Constant output at `floor_mw`. Minimum feasible capacity by construction — no sizing needed. Capacity factor enters only the fuel-cost calculation (annual generation = capacity × CF × 8760).*

In [4]:
kk = KKSupply(smr_tech, demand, prices, downtime_fraction=SMR_DOWNTIME,
              buy_tariff=GRID_TARIFF_BUY,
              grid_connect_annual=grid_connect_annual(demand.demand_mw))

print(f"Installed capacity : {kk.capacity_mw:,.0f} MW")
print(f"Downtime           : {kk.downtime_hours} h/yr  ({SMR_DOWNTIME:.0%} of year)")
print(f"Annual generation  : {kk.capacity_mw * (demand.HOURS - kk.downtime_hours) / 1e6:.3f} TWh")
print(f"Annual on-site cost: {kk.annual_cost() / 1e6:.1f} M€")

Installed capacity : 1,000 MW
Downtime           : 876 h/yr  (10% of year)
Annual generation  : 7.884 TWh
Annual on-site cost: 598.7 M€


## 4. VE — solar + wind + battery on-site
*Single LP over all capacity and dispatch variables simultaneously (HiGHS). `storage_hours` fixes E_batt = storage_hours × P_batt; set to `None` to let E_batt optimise freely. Solves in seconds.*

In [5]:
ve = VESupply(solar_cf, wind_cf, solar_tech, wind_tech, battery, demand,
              prices=prices, storage_hours=4,
              buy_tariff=GRID_TARIFF_BUY, sell_tariff=GRID_TARIFF_SELL,
              grid_connect_annual=grid_connect_annual(demand.grid_cap_mw))

t0 = time.time()
_ = ve.solution   # trigger optimisation
print(f"Optimisation done in {time.time()-t0:.1f} s")
print()
print(f"Solar              : {ve.c_solar:,.1f} MW")
print(f"Wind               : {ve.c_wind:,.1f} MW")
dur = f"{ve.batt_energy_mwh/ve.batt_power_mw:.1f}h" if ve.batt_power_mw > 0 else "—"
print(f"Battery            : {ve.batt_power_mw:,.1f} MW / {ve.batt_energy_mwh:,.1f} MWh  ({dur})")
soc_init = ve.lp_detail()["soc"][-1]
print(f"Battery init SOC   : {soc_init:,.1f} MWh  ({soc_init / ve.batt_energy_mwh * 100:.1f} % of capacity)")
print(f"Annual on-site cost: {ve.annual_onsite_cost() / 1e6:.1f} M€")

ve.save()
ve.save_lp_arrays()
ve.save_lp_txt()
print("Solution saved → runs/ve_solution.json, runs/ve_lp_arrays.npz, runs/lp_arrays/*.txt")

Optimisation done in 22.6 s

Solar              : 6,191.8 MW
Wind               : 3,401.7 MW
Battery            : 7,343.2 MW / 29,372.7 MWh  (4.0h)
Battery init SOC   : 772.0 MWh  (2.6 % of capacity)
Annual on-site cost: 1066.3 M€
Solution saved → runs/ve_solution.json, runs/ve_lp_arrays.npz, runs/lp_arrays/*.txt


## 5. Full model — KK vs VE
*Adds grid purchase costs to each on-site option and computes total annualised cost and system LCOE.*

In [6]:
model = DatacenterModel(demand, grid, kk, ve)
results = model.run()

rows = []
for tech, res in results.items():
    import_cost = res.annual_grid_cost + res.annual_tariff_cost + res.annual_grid_connect
    rows.append({
        "Tech": tech,
        "Inv (M€/yr)":          round(res.annual_inv_cost       / 1e6, 1),
        "O&M (M€/yr)":          round(res.annual_om_cost        / 1e6, 1),
        "Import costs (M€/yr)": round(import_cost               / 1e6, 1),
        "Export rev. (M€/yr)":  round(res.annual_export_revenue / 1e6, 1),
        "Total (M€/yr)":        round(res.annual_total_cost     / 1e6, 1),
        "LCOE (€/MWh)":         round(res.lcoe, 2),
        "Grid import (TWh)":    round(res.grid_import_mwh       / 1e6, 3),
    })

pd.DataFrame(rows).set_index("Tech")

,Inv (M€/yr),O&M (M€/yr),Import costs (M€/yr),Export rev. (M€/yr),Total (M€/yr),LCOE (€/MWh),Grid import (TWh)
Tech,,,,,,,
KK,353.6,245.1,61.7,0.0,660.4,75.39,0.876
VE,842.5,223.8,78.1,256.1,888.3,101.41,0.784


### 5.b  Seasonal analysis — winter vs summer
*Same empirical patterns, sliced to Oct–Mar (winter) and Apr–Sep (summer). Fixed capacity costs (capex×CRF + opex\_fixed) are pro-rated by the season fraction; variable O&M and grid costs fall only on hours in the window. The cyclic SOC constraint closes over the seasonal window (start-of-season SOC = end-of-season SOC).*

In [7]:
# Variation patterns cover 2025-01-01 00:00 … 2025-12-31 23:00 (calendar year 2025)
_dt    = pd.date_range("2025-01-01", periods=8760, freq="h")
WINTER = (_dt.month >= 10) | (_dt.month <= 3)   # Oct–Mar
SUMMER = ~WINTER

for label, mask in [("Winter (Oct–Mar)", WINTER), ("Summer (Apr–Sep)", SUMMER)]:
    print(f"{label}: {mask.sum()} h  ({mask.sum()/8760:.3f} of year)")

Winter (Oct–Mar): 4368 h  (0.499 of year)
Summer (Apr–Sep): 4392 h  (0.501 of year)


In [8]:
from pylib.model import DatacenterDemand

seasonal_results = {}

for label, mask in [("Winter", WINTER), ("Summer", SUMMER)]:
    T_sub  = int(mask.sum())
    sf     = T_sub / 8760

    dem_s  = DatacenterDemand(demand_mw=1_000.0, x=0.50, hours=T_sub)
    grid_s = GridSupply(prices[mask], dem_s)

    kk_s = KKSupply(
        smr_tech, dem_s, prices[mask],
        downtime_fraction   = SMR_DOWNTIME,
        buy_tariff          = GRID_TARIFF_BUY,
        grid_connect_annual = grid_connect_annual(dem_s.demand_mw) * sf,
    )
    ve_s = VESupply(
        solar_cf[mask], wind_cf[mask], solar_tech, wind_tech, battery, dem_s,
        prices              = prices[mask],
        storage_hours       = 4,
        buy_tariff          = GRID_TARIFF_BUY,
        sell_tariff         = GRID_TARIFF_SELL,
        grid_connect_annual = grid_connect_annual(dem_s.grid_cap_mw) * sf,
    )

    t0 = time.time()
    r  = DatacenterModel(dem_s, grid_s, kk_s, ve_s).run()
    print(f"{label} ({T_sub} h, sf={sf:.3f})  —  solved in {time.time()-t0:.1f} s")
    for res in r.values():
        print(f"  {res}")
    print()
    seasonal_results[label] = r

Winter (4368 h, sf=0.499)  —  solved in 6.2 s
  KK [1000 MW SMR] | LCOE 76.40 €/MWh | total 333.7 M€/yr
  VE [3661 MW PV + 4645 MW wind + 7712 MW / 30849 MWh batt] | LCOE 109.11 €/MWh | total 476.6 M€/yr

Summer (4392 h, sf=0.501)  —  solved in 8.8 s
  KK [1000 MW SMR] | LCOE 75.19 €/MWh | total 330.3 M€/yr
  VE [7912 MW PV + 355 MW wind + 3854 MW / 15417 MWh batt] | LCOE 51.08 €/MWh | total 224.3 M€/yr



In [9]:
rows = []
for season, r in seasonal_results.items():
    for tech, res in r.items():
        import_cost = res.annual_grid_cost + res.annual_tariff_cost + res.annual_grid_connect
        rows.append({
            "Season": season, "Tech": tech,
            "Inv (M€/yr)":          round(res.annual_inv_cost       / 1e6, 1),
            "O&M (M€/yr)":          round(res.annual_om_cost        / 1e6, 1),
            "Import costs (M€/yr)": round(import_cost               / 1e6, 1),
            "Export rev. (M€/yr)":  round(res.annual_export_revenue / 1e6, 1),
            "Total (M€/yr)":        round(res.annual_total_cost     / 1e6, 1),
            "LCOE (€/MWh)":         round(res.lcoe, 2),
            "Grid import (TWh)":    round(res.grid_import_mwh       / 1e6, 3),
        })

pd.DataFrame(rows).set_index(["Season", "Tech"])

Inv (M€/yr)  O&M (M€/yr)  Import costs (M€/yr)  \
Season Tech                                                   
Winter KK          176.3        122.2                  35.2   
       VE          444.5        108.3                  56.0   
Summer KK          177.3        122.9                  30.1   
       VE          227.2         79.5                  27.8   

             Export rev. (M€/yr)  Total (M€/yr)  LCOE (€/MWh)  \
Season Tech                                                     
Winter KK                    0.0          333.7         76.40   
       VE                  132.3          476.6        109.11   
Summer KK                    0.0          330.3         75.19   
       VE                  110.1          224.3         51.08   

             Grid import (TWh)  
Season Tech                     
Winter KK                0.437  
       VE                0.483  
Summer KK                0.439  
       VE                0.464